In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/drw-crypto-market-prediction/sample_submission.csv
/kaggle/input/drw-crypto-market-prediction/train.parquet
/kaggle/input/drw-crypto-market-prediction/test.parquet


In [2]:
!rm -rf /kaggle/working/crypto_market_prediction
!git clone https://github.com/CapstoneTeam23UMICH/crypto_market_prediction.git
!pip install -r /kaggle/working/crypto_market_prediction/requirements.txt

Cloning into 'crypto_market_prediction'...
remote: Enumerating objects: 3210, done.
remote: Counting objects: 100% (3079/3079), done.
remote: Compressing objects: 100% (2066/2066), done.
remote: Total 3210 (delta 586), reused 3018 (delta 559), pack-reused 131 (from 1)
Receiving objects: 100% (3210/3210), 28.40 MiB | 35.34 MiB/s, done.
Resolving deltas: 100% (637/637), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 88.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 71.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━

In [3]:
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import shap
from tqdm import tqdm
import warnings
import torch
import torch.nn as nn
import torch.optim as optim

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import gc

sys.path.append('/kaggle/working/crypto_market_prediction')

from src.github_push_file import push_parquet_to_github
from src.github_push_folder import push_folder_to_github

from src.get_refresh_metadata import (
    get_feature_drift_df,
    get_correlation_train_df,
    get_correlation_test_df,
    get_autocorrelation_train_df,
    get_adversarial_validation_df,
    get_mutual_information_train_df,
    get_correlation_stability_df,
    get_vif_train_df)

from src.common import (
    evaluate_regression, 
    evaluate_classification, 
    preprocess_classifier_sae)

from src.get_feature_set import get_feature_set
from src.get_cv_splits import get_folds_for_model
from src.augmented_rfe import augmented_rfe_timeseries_lightgbm

from src.model_registry import (
    model_registry_regression,
    model_registry_autoencoder)

from src.model_fit_predict import (
    fit_predict_tree, 
    fit_predict_mlp, 
    fit_predict_classifier_sae)

from src.run_cv import (
    to_tensor,
    run_cv,
    expand_grid,
    run_grid_search
)

df_drift = get_feature_drift_df()
df_corr_train = get_correlation_train_df()
df_autocorr = get_autocorrelation_train_df()
df_adv_val = get_adversarial_validation_df()
df_mutual_information = get_mutual_information_train_df()
df_corr_stability = get_correlation_stability_df()
df_vif = get_vif_train_df()

Loading existing df_feature_drift.parquet
Loading existing corr_long_train.parquet
Loading existing autocorr_long_train.parquet
Loading existing df_adversarial_validation.parquet
Loading existing df_mutual_information_train.parquet
Loading existing df_corr_stability.parquet
Loading existing df_vif.parquet


In [4]:
train_path = '/kaggle/input/drw-crypto-market-prediction/train.parquet'
test_path = '/kaggle/input/drw-crypto-market-prediction/test.parquet'

df_train = pd.read_parquet(train_path).astype('float32')
df_test = pd.read_parquet(test_path).astype('float32')

In [5]:
known_features = ['bid_qty', 'ask_qty', 'buy_qty', 'sell_qty', 'volume']
target = 'label'
anonymized_features = sorted(list(set(df_train.columns) - set(known_features) - set([target])), key=lambda x: int(x[1:]))

Loading existing df_feature_drift.parquet
Loading existing corr_long_train.parquet
Loading existing autocorr_long_train.parquet
Loading existing df_adversarial_validation.parquet
Loading existing df_mutual_information_train.parquet
Loading existing df_vif.parquet
Returning Features Set of Size: 236


invalid value encountered in less


### Initial Feature Selection

In [7]:
selected_features = get_feature_set(anonymized_features, corr_thresh = 0.99, vif_tresh = 100, mi_tresh = 0.02)

Loading existing df_feature_drift.parquet
Loading existing corr_long_train.parquet
Loading existing autocorr_long_train.parquet
Loading existing df_adversarial_validation.parquet
Loading existing df_mutual_information_train.parquet
Loading existing df_vif.parquet
Returning Features Set of Size: 236


invalid value encountered in less


### Denoising with Classifier Supervised Autoencoder

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df_train_ae = preprocess_classifier_sae(df_train, selected_features + ['label'])
folds_ae = get_folds_for_model('classifier_SAE', df_train_ae)
X_train_ae = to_tensor(df_train_ae.loc[folds_ae[4][0], selected_features], device=device)
w_train_ae = to_tensor(df_train_ae.loc[folds_ae[4][0], ['weight']], device=device)
y_train_ae = to_tensor(df_train_ae.loc[folds_ae[4][0], ['target']], device=device).view(-1, 1)

X_val_ae   = to_tensor(df_train_ae.loc[folds_ae[4][1], selected_features], device=device)
w_val_ae   = to_tensor(df_train_ae.loc[folds_ae[4][1], ['weight']], device=device)
y_val_ae   = to_tensor(df_train_ae.loc[folds_ae[4][1], ['target']], device=device).view(-1, 1)
model_with_param_ae = model_registry_autoencoder(mode='best_model_single_param')['classifier_SAE']

In [3]:

import torch
import torch.nn as nn
class ClassifierSupervisedAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.SiLU(), nn.BatchNorm1d(128),
            nn.Linear(128, 64), nn.SiLU(), nn.BatchNorm1d(64)
        )
        self.bottleneck = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.SiLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 128), nn.SiLU(), nn.BatchNorm1d(128),
            nn.Linear(128, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.SiLU(), nn.BatchNorm1d(64), nn.Dropout(0.5),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x, noise_std=0.0):
        if noise_std > 0:
            x = x + noise_std * torch.randn_like(x)
        encoded = self.encoder(x)
        latent = self.bottleneck(encoded)
        recon = self.decoder(latent)
        out = self.classifier(latent)
        return recon, out, latent

classifier_sae_grid = {
    "latent_dim": [64],
    "lr": [0.006],
    "weight_decay": [0.06],
    "noise_std": [0.01],
    "recon_alpha":[0.5],
    "random_state": [42]
}
model_with_param_ae =  (ClassifierSupervisedAutoencoder, classifier_sae_grid)

In [74]:
_, _, best_model_ae = fit_predict_classifier_sae(model_with_param_ae,
                                                 X_train_ae, 
                                                 y_train_ae, 
                                                 w_train_ae,
                                                 X_val_ae,   
                                                 y_val_ae,   
                                                 w_val_ae,
                                                 max_epochs=1000
                                                 )

In [75]:
X_train = torch.tensor(df_train[selected_features].values, dtype=torch.float32).to(device)
X_test = torch.tensor(df_test[selected_features].values, dtype=torch.float32).to(device)

best_model_ae.eval()
with torch.no_grad():
    _, _, latent_train = best_model_ae(X_train)
    _, _, latent_test = best_model_ae(X_test)

df_latent_test = pd.DataFrame(
    latent_train.cpu().numpy(),
    index=df_train.index,
    columns=[f'latent_{i}' for i in range(latent_train.shape[1])]
)

df_latent_train = pd.concat([df_latent_test, df_train['label']], axis=1)
print("Latent features for df_train extracted:", df_latent_train.shape)

df_latent_test = pd.DataFrame(
    latent_test.cpu().numpy(),
    index=df_test.index,
    columns=[f'latent_{i}' for i in range(latent_test.shape[1])]
)

df_latent_test = pd.concat([df_latent_test, df_test['label']], axis=1)
print("Latent features for df_test extracted:", df_latent_test.shape)

Latent features for df_train extracted: (525886, 65)
Latent features for df_test extracted: (538150, 65)


### Augmented Feature Elimination on Latent Features

In [121]:
arfe_features, arfe_feature_history = augmented_rfe_timeseries_lightgbm(
    df_latent_train,
    label_col='label',
    model_params=None,
    n_splits=5,
    num_boost_round=100,
    min_features=50,
    fold_weights=[1, 1, 1, 2, 3],
    verbose=True
)

Round 0 | Fold 0 | Val Pearson: 0.1680
Round 0 | Fold 1 | Val Pearson: 0.2051
Round 0 | Fold 2 | Val Pearson: 0.1891
Round 0 | Fold 3 | Val Pearson: 0.2509
Round 0 | Fold 4 | Val Pearson: 0.0740

Round 0 | Features: 64
Trying to drop: latent_15
Round 0 | Fold 0 | Val Pearson: 0.1698
Round 0 | Fold 1 | Val Pearson: 0.2034
Round 0 | Fold 2 | Val Pearson: 0.1862
Round 0 | Fold 3 | Val Pearson: 0.2494
Round 0 | Fold 4 | Val Pearson: 0.0743
Dropped: latent_15 (Weighted improvement = 4)

Round 1 | Features: 63
Trying to drop: latent_50
Round 1 | Fold 0 | Val Pearson: 0.1698
Round 1 | Fold 1 | Val Pearson: 0.2041
Round 1 | Fold 2 | Val Pearson: 0.1895
Round 1 | Fold 3 | Val Pearson: 0.2485
Round 1 | Fold 4 | Val Pearson: 0.0744
Dropped: latent_50 (Weighted improvement = 6)

Round 2 | Features: 62
Trying to drop: latent_29
Round 2 | Fold 0 | Val Pearson: 0.1744
Round 2 | Fold 1 | Val Pearson: 0.2034
Round 2 | Fold 2 | Val Pearson: 0.1893
Round 2 | Fold 3 | Val Pearson: 0.2473
Round 2 | Fold 4 

### Train Best MLP Regression Model

In [123]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
start = '2022-4-01'
end = '2023-10-01'

X_train_mlp = to_tensor(df_latent_train.loc[start:end, arfe_features], device=device)
y_train_mlp = to_tensor(df_latent_train.loc[start:end, ['label']], device=device).view(-1, 1)

X_test_mlp  = to_tensor(df_latent_test.loc[:, arfe_features], device=device)
y_test_mlp = to_tensor(df_latent_test.loc[:, ['label']], device=device).view(-1, 1)
model_with_param_mlp = model_registry_regression(mode='best_model_single_param')['MLP']

In [126]:
_, y_test_pred, best_model_mlp = fit_predict_mlp(model_with_param_mlp,
                                                X_train_mlp, 
                                                y_train_mlp,
                                                X_test_mlp,   
                                                y_test_mlp,
                                                max_epochs = 500
                                                )

### Prediction & Submission

In [127]:
submission_path = '/kaggle/working/submission.csv'
submission = pd.DataFrame({
    'ID': df_test.index,
    'prediction': y_test_pred
})
submission.to_csv(submission_path, index=False)